# Improvement 2: MIDI-Guided Post-Processing Audio Masking

## Problem

The separation model (especially for Guitar 2) often generates **hallucinated noise**
during time regions where the target guitar is actually silent. The model perceives
the other guitar's signal as faint background and fails to fully suppress it,
leading to low SDR/SAR scores.

## Solution: Score-Informed Audio Masking

After separation, we use the **per-guitar note transcription** (from Basic Pitch +
pitch splitting, or from ground-truth annotations) as a binary mask:

- When Guitar 2's transcription indicates **no notes are playing** in a time window,
  we force Guitar 2's separated audio to zero (or apply a smooth fade envelope)
- This eliminates hallucinated leakage without touching the neural network weights

### Soft masking with fade envelope

Instead of hard binary muting (which causes clicks), we apply a **smooth fade**
envelope at the boundaries of active regions. This uses a short linear ramp
(default 10ms) to transition between masked (silent) and unmasked (active) regions.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path(".").resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import torch
import torchaudio
import IPython.display as ipd
import museval

from src.models.factory import build_model
from src.models.apply import apply_model
from src.data.manifests import load_manifest
from src.inference.separate import create_tensor_for_segment
from src.utils.io import load_config

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

## 1. MIDI-Guided Masking Implementation

In [ ]:
def build_activity_mask(piano_roll_128, sr=44100, fade_ms=10):
    """Build a smooth activity mask from a single-guitar 128-note piano roll.

    Args:
        piano_roll_128: (128, T) binary piano roll for one guitar
        sr: sample rate
        fade_ms: ramp duration at onset/offset boundaries (ms)

    Returns:
        mask: (T,) float tensor in [0, 1], 1 = keep audio, 0 = mute
    """
    T = piano_roll_128.shape[-1]

    # Any pitch active at frame t → guitar is playing
    activity = (piano_roll_128.sum(dim=0) > 0).float()  # (T,)

    # Apply fade envelope at boundaries
    fade_samples = int(fade_ms * sr / 1000)
    if fade_samples < 1:
        return activity

    # Use a 1D convolution to smooth the edges
    kernel = torch.ones(1, 1, fade_samples * 2 + 1) / (fade_samples * 2 + 1)
    padded = activity.unsqueeze(0).unsqueeze(0)  # (1, 1, T)
    smoothed = torch.nn.functional.conv1d(
        padded, kernel, padding=fade_samples
    ).squeeze()[:T]

    # Clamp to [0, 1] and ensure active regions stay at 1
    mask = torch.clamp(smoothed, 0.0, 1.0)
    mask = torch.maximum(mask, activity)

    return mask


def apply_midi_masking(g1_audio, g2_audio, piano_roll, sr=44100, fade_ms=10):
    """Apply MIDI-guided masking to separated audio.

    Args:
        g1_audio: (channels, T) Guitar 1 separated audio
        g2_audio: (channels, T) Guitar 2 separated audio
        piano_roll: (256, T) split piano roll — [0:128] = G1, [128:256] = G2
        sr: sample rate
        fade_ms: smooth ramp at mask edges

    Returns:
        g1_masked, g2_masked: masked audio tensors
    """
    T_audio = min(g1_audio.shape[-1], g2_audio.shape[-1])
    T_notes = piano_roll.shape[-1]
    T = min(T_audio, T_notes)

    g1_roll = piano_roll[:128, :T]
    g2_roll = piano_roll[128:, :T]

    mask_g1 = build_activity_mask(g1_roll, sr=sr, fade_ms=fade_ms)
    mask_g2 = build_activity_mask(g2_roll, sr=sr, fade_ms=fade_ms)

    g1_masked = g1_audio[:, :T] * mask_g1.unsqueeze(0)
    g2_masked = g2_audio[:, :T] * mask_g2.unsqueeze(0)

    muted_g1 = (mask_g1 == 0).sum().item()
    muted_g2 = (mask_g2 == 0).sum().item()
    print(f"  G1: muted {muted_g1/sr:.2f}s of {T/sr:.2f}s ({100*muted_g1/T:.1f}%)")
    print(f"  G2: muted {muted_g2/sr:.2f}s of {T/sr:.2f}s ({100*muted_g2/T:.1f}%)")

    return g1_masked, g2_masked, mask_g1, mask_g2


print("MIDI masking functions defined.")

## 2. Load models

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

def load_model(config_path, checkpoint_path, device):
    config = load_config(config_path)
    model = build_model(config["model"]["name"], config["model"].get("kwargs", {}))
    if checkpoint_path.exists():
        payload = torch.load(checkpoint_path, map_location=device, weights_only=False)
        state = payload.get("model_state_dict", payload)
        model.load_state_dict(state)
        print(f"Loaded: {checkpoint_path.name}")
    else:
        print(f"WARNING: {checkpoint_path} not found")
    model.to(device).eval()
    return model, config

model_cond,   cfg_cond   = load_model(REPO_ROOT / "configs" / "conditioned.yaml",   REPO_ROOT / "outputs" / "checkpoints" / "best_conditioned.pt",   device)
model_uncond, cfg_uncond = load_model(REPO_ROOT / "configs" / "unconditioned.yaml", REPO_ROOT / "outputs" / "checkpoints" / "best_unconditioned.pt", device)

## 3. Select a test track

In [ ]:
manifest = load_manifest(REPO_ROOT / cfg_cond["dataset"]["manifest"], resolve_root=REPO_ROOT)
test_entries = [e for e in manifest if e["split"] == cfg_cond["dataset"]["test_split"]]
print(f"Test tracks: {len(test_entries)}")
for i, e in enumerate(test_entries):
    print(f"  [{i}] {e['track_name']}")

TRACK_IDX = 0  # <-- change this
entry = test_entries[TRACK_IDX]
print(f"\nSelected: {entry['track_name']}")

mix, sr = torchaudio.load(entry["mix"])
g1_ref, _ = torchaudio.load(entry["sources"]["guitar1"])
g2_ref, _ = torchaudio.load(entry["sources"]["guitar2"])
print(f"Duration: {mix.shape[-1]/sr:.2f}s")

## 4. Separate (baseline — no masking)

In [ ]:
def separate_track(model, mix, entry, device):
    ref = mix.mean(0)
    ref_mean, ref_std = ref.mean(), ref.std()
    normalized = (mix - ref_mean) / ref_std

    with torch.no_grad():
        if getattr(model, "note_conditioning", False):
            notes = create_tensor_for_segment(
                entry["notes_csv"], segment_start=0, segment_end=mix.shape[-1],
            )
            model_input = torch.cat((normalized, notes), dim=0)
            sources = apply_model(model, model_input[None], progress=False, device=device)[0]
        else:
            sources = apply_model(model, normalized[None], progress=False, device=device)[0]

    sources = sources * ref_std + ref_mean
    return sources[0].cpu(), sources[1].cpu()


print("Separating with conditioned model...")
g1_cond, g2_cond = separate_track(model_cond, mix, entry, device)

print("Separating with unconditioned model...")
g1_uncond, g2_uncond = separate_track(model_uncond, mix, entry, device)

print("Done.")

## 5. Build the note roll for masking

Here we use the ground-truth `notes.csv` when available for a clean demonstration.
In production, you would use the Basic Pitch + pitch splitting pipeline from
notebook 05.

In [ ]:
# Use ground-truth notes if available, otherwise fall back to Basic Pitch
if entry.get("notes_csv") and Path(entry["notes_csv"]).exists():
    print("Using ground-truth notes.csv for masking")
    notes_roll = create_tensor_for_segment(
        entry["notes_csv"], segment_start=0, segment_end=mix.shape[-1],
    )
else:
    print("No ground-truth notes — using Basic Pitch + pitch splitting")
    from basic_pitch.inference import predict as bp_predict

    # Import the splitting function from notebook 05's logic
    def median_split_with_hysteresis(note_events, audio_length_samples, sr=44100,
                                     window_sec=0.5, hysteresis_semitones=2):
        T = audio_length_samples
        pitch_sum = np.zeros(T, dtype=np.float64)
        pitch_count = np.zeros(T, dtype=np.float64)
        for ev in note_events:
            s, e_t, pitch = int(ev[0]*sr), min(int(ev[1]*sr), T), int(ev[2])
            if s < e_t:
                pitch_sum[s:e_t] += pitch
                pitch_count[s:e_t] += 1
        win = max(1, int(window_sec * sr))
        kernel = np.ones(win) / win
        sm_p = np.convolve(pitch_sum, kernel, mode="same")
        sm_c = np.maximum(np.convolve(pitch_count, kernel, mode="same"), 1e-8)
        running_med = sm_p / sm_c
        global_med = float(np.median([int(ev[2]) for ev in note_events])) if note_events else 60.0
        running_med[sm_p == 0] = global_med
        g1 = np.zeros((128, T), dtype=np.float32)
        g2 = np.zeros((128, T), dtype=np.float32)
        last_asgn = {}
        for ev in sorted(note_events, key=lambda x: x[0]):
            s, e_t, pitch = int(ev[0]*sr), min(int(ev[1]*sr), T), int(ev[2])
            if s >= e_t or pitch < 0 or pitch >= 128: continue
            diff = pitch - running_med[min(s, T-1)]
            if abs(diff) <= hysteresis_semitones:
                a = last_asgn.get(pitch, 1 if diff >= 0 else 2)
            else:
                a = 1 if diff > 0 else 2
            last_asgn[pitch] = a
            (g1 if a == 1 else g2)[pitch, s:e_t] = 1.0
        return torch.from_numpy(np.concatenate([g1, g2], axis=0))

    _, _, note_events = bp_predict(str(entry["mix"]))
    notes_roll = median_split_with_hysteresis(note_events, mix.shape[-1], sr=sr)

print(f"Notes roll shape: {notes_roll.shape}")
print(f"G1 active frames: {(notes_roll[:128].sum(0) > 0).sum().item()}")
print(f"G2 active frames: {(notes_roll[128:].sum(0) > 0).sum().item()}")

## 6. Apply MIDI masking

In [ ]:
print("Masking conditioned output:")
g1_cond_m, g2_cond_m, mask_g1, mask_g2 = apply_midi_masking(
    g1_cond, g2_cond, notes_roll, sr=sr, fade_ms=10,
)

print("\nMasking unconditioned output:")
g1_uncond_m, g2_uncond_m, _, _ = apply_midi_masking(
    g1_uncond, g2_uncond, notes_roll, sr=sr, fade_ms=10,
)

## 7. Visualize the masks

In [ ]:
T_vis = min(mask_g1.shape[0], mask_g2.shape[0])
time_axis = np.arange(T_vis) / sr

fig, axes = plt.subplots(2, 1, figsize=(16, 4), sharex=True)

axes[0].fill_between(time_axis, mask_g1[:T_vis].numpy(), alpha=0.4, color="tab:blue")
axes[0].set_ylabel("G1 mask")
axes[0].set_ylim(-0.05, 1.1)
axes[0].set_title("Activity masks (1 = keep, 0 = mute)")

axes[1].fill_between(time_axis, mask_g2[:T_vis].numpy(), alpha=0.4, color="tab:orange")
axes[1].set_ylabel("G2 mask")
axes[1].set_ylim(-0.05, 1.1)
axes[1].set_xlabel("Time (s)")

plt.tight_layout()
plt.show()

## 8. Metrics comparison: before vs after masking

In [ ]:
def compute_bss_metrics(g1_ref, g2_ref, g1_est, g2_est, sr):
    min_len = min(g1_ref.shape[-1], g1_est.shape[-1])
    refs = np.stack([
        g1_ref[:, :min_len].T.numpy(),
        g2_ref[:, :min_len].T.numpy(),
    ], axis=0)
    ests = np.stack([
        g1_est[:, :min_len].T.numpy(),
        g2_est[:, :min_len].T.numpy(),
    ], axis=0)
    sdr, sir, isr, sar, _ = museval.metrics.bss_eval(
        refs, ests, compute_permutation=True,
        window=sr, hop=sr,
        framewise_filters=False, bsseval_sources_version=False,
    )
    return {
        "Guitar 1": {"SDR": float(np.nanmedian(sdr[0])), "SIR": float(np.nanmedian(sir[0])), "SAR": float(np.nanmedian(sar[0]))},
        "Guitar 2": {"SDR": float(np.nanmedian(sdr[1])), "SIR": float(np.nanmedian(sir[1])), "SAR": float(np.nanmedian(sar[1]))},
    }


m_cond_raw    = compute_bss_metrics(g1_ref, g2_ref, g1_cond,     g2_cond,     sr)
m_cond_masked = compute_bss_metrics(g1_ref, g2_ref, g1_cond_m,   g2_cond_m,   sr)
m_unc_raw     = compute_bss_metrics(g1_ref, g2_ref, g1_uncond,   g2_uncond,   sr)
m_unc_masked  = compute_bss_metrics(g1_ref, g2_ref, g1_uncond_m, g2_uncond_m, sr)

print(f"{'':20s} {'Cond Raw':>10s} {'Cond Masked':>12s} {'Unc Raw':>10s} {'Unc Masked':>12s}")
print("-" * 68)
for source in ["Guitar 1", "Guitar 2"]:
    for metric in ["SDR", "SIR", "SAR"]:
        cr = m_cond_raw[source][metric]
        cm = m_cond_masked[source][metric]
        ur = m_unc_raw[source][metric]
        um = m_unc_masked[source][metric]
        label = f"{source} {metric}"
        print(f"{label:20s} {cr:+8.2f}   {cm:+8.2f}      {ur:+8.2f}   {um:+8.2f}")

## 9. Audio comparison

In [ ]:
min_len = min(mix.shape[-1], g1_cond_m.shape[-1], g1_uncond_m.shape[-1])

print("=== Mix ===")
ipd.display(ipd.Audio(mix[:, :min_len].numpy(), rate=sr))

for guitar, ref, raw_c, masked_c, raw_u, masked_u in [
    ("Guitar 1", g1_ref, g1_cond, g1_cond_m, g1_uncond, g1_uncond_m),
    ("Guitar 2", g2_ref, g2_cond, g2_cond_m, g2_uncond, g2_uncond_m),
]:
    print(f"\n=== {guitar} — Reference ===")
    ipd.display(ipd.Audio(ref[:, :min_len].numpy(), rate=sr))
    print(f"=== {guitar} — Conditioned (raw) ===")
    ipd.display(ipd.Audio(raw_c[:, :min_len].numpy(), rate=sr))
    print(f"=== {guitar} — Conditioned + MIDI Masking ===")
    ipd.display(ipd.Audio(masked_c[:, :min_len].numpy(), rate=sr))
    print(f"=== {guitar} — Unconditioned (raw) ===")
    ipd.display(ipd.Audio(raw_u[:, :min_len].numpy(), rate=sr))
    print(f"=== {guitar} — Unconditioned + MIDI Masking ===")
    ipd.display(ipd.Audio(masked_u[:, :min_len].numpy(), rate=sr))

## 10. Spectrogram: Before vs After masking

In [ ]:
def plot_spectrogram(wav, sr, ax, title="", vmin=-60, vmax=0):
    mono = wav.mean(dim=0)[:min_len]
    spec = torch.stft(mono, n_fft=2048, hop_length=512,
                      window=torch.hann_window(2048), return_complex=True)
    mag_db = 20 * torch.log10(spec.abs().clamp(min=1e-8))
    ax.imshow(mag_db.numpy(), aspect="auto", origin="lower",
              extent=[0, min_len/sr, 0, sr/2], cmap="magma", vmin=vmin, vmax=vmax)
    ax.set_ylim(0, 8000)
    ax.set_title(title, fontsize=9)


fig, axes = plt.subplots(2, 3, figsize=(18, 8), sharex=True, sharey=True)

plot_spectrogram(g2_ref,       sr, axes[0, 0], "G2 — Reference")
plot_spectrogram(g2_cond,      sr, axes[0, 1], "G2 — Conditioned (raw)")
plot_spectrogram(g2_cond_m,    sr, axes[0, 2], "G2 — Conditioned + Masking")
plot_spectrogram(g2_ref,       sr, axes[1, 0], "G2 — Reference")
plot_spectrogram(g2_uncond,    sr, axes[1, 1], "G2 — Unconditioned (raw)")
plot_spectrogram(g2_uncond_m,  sr, axes[1, 2], "G2 — Unconditioned + Masking")

for ax in axes[-1]:
    ax.set_xlabel("Time (s)")
for ax in axes[:, 0]:
    ax.set_ylabel("Freq (Hz)")

plt.suptitle(f"Improvement 2: MIDI Masking Effect on Guitar 2 — {entry['track_name']}", fontsize=13)
plt.tight_layout()
plt.show()

## 11. Batch evaluation

In [ ]:
results_raw = []
results_masked = []

for i, entry_i in enumerate(test_entries):
    print(f"[{i+1}/{len(test_entries)}] {entry_i['track_name']}", end=" ")
    mix_i, sr_i = torchaudio.load(entry_i["mix"])
    g1r_i, _ = torchaudio.load(entry_i["sources"]["guitar1"])
    g2r_i, _ = torchaudio.load(entry_i["sources"]["guitar2"])

    try:
        g1c, g2c = separate_track(model_cond, mix_i, entry_i, device)

        if entry_i.get("notes_csv") and Path(entry_i["notes_csv"]).exists():
            nr = create_tensor_for_segment(entry_i["notes_csv"], 0, mix_i.shape[-1])
        else:
            print("(no notes) ", end="")
            results_raw.append(None)
            results_masked.append(None)
            print("skip")
            continue

        results_raw.append(compute_bss_metrics(g1r_i, g2r_i, g1c, g2c, sr_i))

        g1m, g2m, _, _ = apply_midi_masking(g1c, g2c, nr, sr=sr_i, fade_ms=10)
        results_masked.append(compute_bss_metrics(g1r_i, g2r_i, g1m, g2m, sr_i))
        print("ok")
    except Exception as exc:
        print(f"FAILED: {exc}")
        results_raw.append(None)
        results_masked.append(None)

print("\n=== Aggregate: Conditioned Raw vs Conditioned + MIDI Masking ===")
print(f"{'':20s} {'Raw':>10s} {'Masked':>10s} {'Delta':>8s}")
print("-" * 52)
for source in ["Guitar 1", "Guitar 2"]:
    for metric in ["SDR", "SIR", "SAR"]:
        vals_r = [m[source][metric] for m in results_raw if m is not None]
        vals_m = [m[source][metric] for m in results_masked if m is not None]
        med_r = np.nanmedian(vals_r) if vals_r else float("nan")
        med_m = np.nanmedian(vals_m) if vals_m else float("nan")
        delta = med_m - med_r
        label = f"{source} {metric}"
        print(f"{label:20s} {med_r:+8.2f}   {med_m:+8.2f}   {delta:+.2f}")